# UPSC ESSAY EVALUATION 


In [125]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
load_dotenv()
import operator
import json

llm1= HuggingFaceEndpoint(
    model="Qwen/Qwen3-235B-A22B",
    task="text-generation",
    temperature=0.5
)
llm2= HuggingFaceEndpoint(
    model="Qwen/Qwen3-8B",
    task="text-generation",
    temperature=0.5
)
llm3= HuggingFaceEndpoint(
    model="Qwen/Qwen3-32B",
    task="text-generation",
    temperature=0.5
)
llm4= HuggingFaceEndpoint(
    model="Qwen/Qwen3-235B-A22B-Thinking-2507",
    task="text-generation",
    temperature=0.5
)
model1= ChatHuggingFace(llm=llm1)
model2= ChatHuggingFace(llm=llm2)
model3= ChatHuggingFace(llm=llm3)
model4= ChatHuggingFace(llm=llm4)





In [126]:
essay = "Climate change is the defining crisis of our generation. Rising temperatures, melting glaciers, and extreme weather events are clear indicators that human activity is disrupting the planet's natural balance. Governments, corporations, and individuals all share responsibility for reducing carbon emissions. Transitioning to renewable energy, adopting sustainable practices, and investing in green technology are no longer optional — they are urgent necessities for survival."

In [127]:
class EvaluationSchema(BaseModel):
    feedback : str = Field (description = "Detalied Feedback for the essay")
    score : int = Field (description = "score out of 10 ")
structured_model = model1.with_structured_output(EvaluationSchema,method="json_mode")

In [128]:
class UPSC_State(TypedDict):
    essay : str
    cot_feedback : str     # clarity of thought cot
    analysis_feedback : str  # analysis of the essay
    language_feedback : str  # language feedback
    overall_feedback : str  # overall feedback
    individual_score :Annotated[list[int],operator.add]
    avg_score : float
    

In [129]:
def evaluate_thought(state:UPSC_State):
    prompt = f'Analyze the following essay and respond ONLY in valid JSON,Evaluate the clarity of thought of the folloeing essay and a provide a feedback and assign score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    return{'cot_feedback':output.feedback,'individual_score':[output.score]}

def evaluate_analysis(state:UPSC_State):
    prompt = f'Analyze the following essay and respond ONLY in valid JSON,Evaluate the depth of analysis of the folloeing essay and a provide a feedback and assign score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    return{'analysis_feedback':output.feedback,'individual_score':[output.score]}

def evaluate_langguage(state:UPSC_State):
    prompt = f'Analyze the following essay and respond ONLY in valid JSON,Evaluate the langguage quality of the following essay and a provide a feedback and assign score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)
    return{'langguage_feedback':output.feedback,'individual_score':[output.score]}

def final_evaluation(state:UPSC_State):
    prompt = f'Analyze the following essay and respond ONLY in valid JSON,Based on following feedbacks create a summarized feedback in langguage feedback -{state["language_feedback"]} \n depth of analysis feedback - {state["analysis_feedback"]}\n clarity of thought feedback - {state["cot_feedback"]}'
    final_feedback = structured_model.invoke(prompt).content
    return{'overall_feedback':final_feedback,'individual_score':[final_feedback.score]}

In [130]:
graph = StateGraph(UPSC_State)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_langguage',evaluate_langguage)
graph.add_node('final_evaluation',final_evaluation)



In [131]:
# edges
graph.add_edge(START,'evaluate_thought')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_langguage')
graph.add_edge('evaluate_thought','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_langguage','final_evaluation')
graph.add_edge('evaluate_thought',END)


In [132]:
workflow = graph.compile()
intial_state = {
    'essay':essay
}
result =workflow.invoke(intial_state)
print(result)

AttributeError: 'dict' object has no attribute 'feedback'